In [ ]:
import numpy as np
m = 1.99e-23 / 1000 # g
kB = 8.31 / 6.02e23
np.sqrt(2  *  m  *  kB  *  1e7)/1.6e-19

np.float64(0.014649534323554793)

In [ ]:
import numpy as np

c = 2.99792458e10
h = 6.62606957e-27
hbar = h / (2 * np.pi)
kB = 1.3806488e-16
me = 9.10938291e-28
mp = 1.672621777e-24
mH = 1.673532499e-24
sigma = 5.670373e-5
statCe = 4.80320425e-10
GeV = 1.602176565e-3
Msolar = 1.989e33
km2cm = 1e5


def loadcsv(path):
    d = np.genfromtxt(path, delimiter=',', names=True)
    r = d['radius']  *  km2cm
    rho = d['density']
    m = d['mass']  *  Msolar
    T = d['temperature']
    return r, rho, m, T


def fermienergy(rho, Ye):
    ne = rho  *  Ye / mp
    pF = hbar  *  (3 * np.pi ** 2 * ne) ** (1/3)
    x = pF / (me * c)
    return me * c ** 2 * (np.sqrt(1+x ** 2)-1) / GeV


def getelecint(T, rho, Ye):
    n = rho / (mp/Ye)
    if n <= 0 or T <= 0:
        return 0.0
    a = 8 * np.pi * me ** 3 * c ** 3/(3 * h ** 3)
    x0 = (n/a) ** (1/3)
    theta = kB * T/(me * c ** 2)
    if theta > x0:
        Ed = 1.5 * n * kB * T
    else:
        eps = np.pi ** 2 * theta ** 2 * (2 * x0 ** 2+1)/(2 * x0 ** 4)
        x = x0 * (1-eps/3.0)
        f = x * (2 * x ** 2-3) * np.sqrt(x ** 2+1) + 3 * np.arcsinh(x)
        g = 8 * x ** 3 * (np.sqrt(x ** 2+1)-1) - f
        A = np.pi * me ** 4 * c ** 5/(3 * h ** 3)
        Ed = A * g * (1 + 4 * np.pi ** 2 * theta ** 2 * ((3 * x ** 2+1) * np.sqrt(x ** 2+1)-(2 * x ** 2+1))/(x * g))
    return (Ed/n) / GeV


def degmfp(T, rho, Z, A, Ye):
    opacity = (56/(15 * np.sqrt(3))) * (statCe ** 6/(c * h * kB ** 2)) * (Z ** 2/(mH * A))/T ** 2
    lR = (np.log((20 * np.sqrt(3)/14) * ((c * h * sigma * kB ** 2 * mH/statCe ** 6) * (A/Z ** 2)))
          + 5 * np.log(T) - np.log(rho))
    a0 = 5.55e-2 * rho ** (1/3)
    I1 = 2 * np.pi * np.log(1+a0 ** 2)
    mu = mp/(mH * Ye)
    lT = np.log((np.pi/8) * ((h ** 3 * kB ** 2)/(statCe ** 4 * me ** 2 * mH))/(mu * Z) * (T * rho/I1))
    kappaeff = opacity/(1+np.exp(lT-lR))
    return 1.0/(kappaeff * rho)


def nondegmfp(T, rho, Z, A, Ye):
    T1, T2 = 1.0128, 1.0823
    a0 = 8.45e-7 * T/rho ** (1/3)
    I1 = 2 * np.pi * np.log(1+a0 ** 2)
    mu = mp/(mH * Ye)
    opacity = (8 * np.pi ** 2 * T2 * statCe ** 6 * h ** 2 * Z ** 2 * rho) / (315 * np.sqrt(3) * T1 * c * (2 * np.pi * me) ** 1.5 * mH ** 2 * kB ** 3.5 * A * mu * T ** 3.5)
    lR = (105 * np.sqrt(3) * T1 * c ** 2 * (2 * np.pi * me) ** 1.5 * sigma * mH ** 2 * kB ** 3.5 * A * mu * T ** 6.5) / (2 * np.pi ** 2 * T2 * statCe ** 6 * h ** 2 * (Z * rho) ** 2)
    lC = (2 ** 6.5 * kB ** 3.5 * T ** 2.5) / (np.sqrt(np.pi) * statCe ** 4 * np.sqrt(me) * Z * I1)
    kappaeff = opacity/(1+lC/lR)
    return 1.0/(kappaeff * rho)


def meanfreepath(T, rho, Z, A, Ye):
    if T/((rho * 1000) ** (2/3)) > 1241:
        return nondegmfp(T, rho, Z, A, Ye)
    return degmfp(T, rho, Z, A, Ye)


def GammaT(rho, mue=2.0):
    a = 8 * np.pi * me ** 3 * c ** 3/(3 * h ** 3)
    x = (rho/(mue * mH)/a) ** (1/3)
    return (4 * x ** 2+5)/(3 * (x ** 2+1))


def callL(source, r, m, rho, T, Ye):
    return source.luminosity(r=r, m=m, rho=rho, T=T,
                             Efermi=fermienergy(rho, Ye),
                             Eelint=getelecint(T, rho, Ye))


def dlnLdlnrho(source, r, m, rho, T, Ye, frac=1e-4):
    L1 = callL(source, r, m, rho * (1+frac), T, Ye)
    L2 = callL(source, r, m, rho * (1-frac), T, Ye)
    if L1 <= 0 or L2 <= 0:
        return 0.0
    return (np.log(L1)-np.log(L2)) / (np.log(1+frac)-np.log(1-frac))


def dlnLdlnT(source, r, m, rho, T, Ye, frac=1e-4):
    L1 = callL(source, r, m, rho, T * (1+frac), Ye)
    L2 = callL(source, r, m, rho, T * (1-frac), Ye)
    if L1 <= 0 or L2 <= 0:
        return 0.0
    return (np.log(L1)-np.log(L2)) / (np.log(1+frac)-np.log(1-frac))


def stability(csvpath, source, Z=6, A=12, Ye=0.5, xi=1.0):

    r, rho, m, T = loadcsv(csvpath)

    GT = GammaT(rho)

    L = np.array([callL(source, ri, mi, rhoi, Ti, Ye)
                 for ri, mi, rhoi, Ti in zip(r, m, rho, T)])
    eps = np.clip(np.gradient(L, m), 0, None)

    alpha = np.array([dlnLdlnrho(source, ri, mi, rhoi, Ti, Ye)
                      for ri, mi, rhoi, Ti in zip(r, m, rho, T)])
    beta = np.array([dlnLdlnT(source, ri, mi, rhoi, Ti, Ye)
                     for ri, mi, rhoi, Ti in zip(r, m, rho, T)])

    kappa = np.array([1.0/(meanfreepath(Ti, rhoi, Z, A, Ye) * rhoi)
                      for Ti, rhoi in zip(T, rho)])
    alphak = np.gradient(np.log(kappa), np.log(rho))
    betak  = np.gradient(np.log(kappa), np.log(T))

    dTdm = np.gradient(T, m)
    dGT1Tdm = np.gradient((GT-1) * T, m)
    with np.errstate(divide='ignore', invalid='ignore'):
        gradterm = np.where(np.abs(dTdm) > 1e-30, dGT1Tdm/dTdm, 0.0)

    Phi = (2 + 3 * (alphak + betak * (GT-1)) - 9 * (GT-1) - 3 * gradterm)
    dPhidm = np.gradient(Phi, m)

    term1 = -3 * xi * eps * (alpha + beta * (GT-1))
    term2 = -xi * eps * Phi
    term3 = -xi * L * dPhidm

    bracket = term1 + term2 + term3
    integrand = (GT-1) * (-3 * xi) * bracket
    I = np.trapezoid(integrand, m) / xi ** 2

    return {
        'integral': I, 'isstable': I < 0,
        'r': r, 'm': m, 'T': T, 'rho': rho, 'GT': GT, 'eps': eps,
        'alpha': alpha, 'beta': beta,
        'alphak': alphak, 'betak': betak,
        'Phi': Phi, 'term1': term1, 'term2': term2, 'term3': term3,
        'integrand': integrand,
    }


from WhiteDwarf.source import singlePBH

path = "/Users/caritsang/Desktop/FYP/SinglePBH/result/PBHM3.0e+14/N2.8e+09/rho01.000e+03.csv"
src = singlePBH(pbhM=3e14, N=2.8e9)

dic = stability(path, src)
print(f"integral = {dic['integral']:.4e}   stable = {dic['isstable']}")

Hawking temperature of the PBH is 4.092e+11
integral = -2.7009e+31   stable = True
